# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [3]:
# My lane is a ranking problem ("which content should an editor refresh first"), not a plain yes/no classification. Per the skill, ranking problems should use a classifier's probability output, evaluated at precision@K, rather than the model's binary label. Logistic Regression is the right starting point because it's simple and readable. If it underperforms the baseline or a stronger model earns its complexity, Random Forest is the natural next step.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [8]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining"] = (df["trend_direction"] == "down").astype(int)

print(df.shape)

from sklearn.model_selection import GroupShuffleSplit

# Grouped split by client_id — no client appears in both train and test.
# This matters because clients have different baselines (established in Week 3's
# frame), so a random row-level split would let the model see each client's
# pattern in both train and test, inflating performance dishonestly.

splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(splitter.split(df, groups=df["client_id"]))

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

print("Train clients:", train_df["client_id"].nunique(), "| Test clients:", test_df["client_id"].nunique())
print("Overlap (should be 0):", len(set(train_df["client_id"]) & set(test_df["client_id"])))
print("Train rows:", len(train_df), "| Test rows:", len(test_df))

(30000, 45)
Train clients: 24 | Test clients: 8
Overlap (should be 0): 0
Train rows: 22885 | Test rows: 7115


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [11]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd

# Feature columns — numeric/tier signals only, no banned columns
feature_cols = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d",
    "users_90d", "engaged_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions",
    "content_age_days", "days_since_last_update",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
]
# NOTE: trend_pct, trend_direction, content_id, client_id deliberately excluded (leakage/ID)
# NOTE: last_30d / prev_30d pairs ALSO excluded — trend_direction/trend_pct were almost
# certainly derived by comparing these two windows, so keeping both lets the model
# reconstruct the label indirectly (leakage one step removed). The _90d trailing
# aggregates are kept since they're a broader summary, not the exact before/after split.

X_train = train_df[feature_cols].fillna(0)
y_train = train_df["is_declining"]
X_test = test_df[feature_cols].fillna(0)
y_test = test_df["is_declining"]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train_scaled, y_train)

model_scores = model.predict_proba(X_test_scaled)[:, 1]

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

# Recompute baseline score on the SAME test split, for a fair comparison
test_stale = (test_df["days_since_last_update"] >= 31).astype(int)
test_expected_ctr = test_df.groupby("position_tier")["ctr"].transform("mean")
test_weak_ctr = (test_df["ctr"] < test_expected_ctr).astype(int)
baseline_scores = test_stale * test_weak_ctr * test_df["impressions_last_30d"]

results = []
for k in [10, 20, 50]:
    results.append({
        "k": k,
        "baseline_precision": precision_at_k(baseline_scores, y_test, k),
        "model_precision": precision_at_k(model_scores, y_test, k),
        "base_rate": y_test.mean(),
    })

comparison_table = pd.DataFrame(results)
print(comparison_table)

coef_table = pd.DataFrame({
    "feature": feature_cols,
    "coefficient": model.coef_[0]
}).sort_values("coefficient", key=abs, ascending=False)

print("Top 10 features by |coefficient| (standardized):")
print(coef_table.head(10).to_string(index=False))

test_df = test_df.copy()
test_df["model_score"] = model_scores
test_df["predicted"] = (test_df["model_score"] >= 0.5).astype(int)
test_df["correct"] = (test_df["predicted"] == test_df["is_declining"])

print("\nError rate by content_type:")
print(test_df.groupby("content_type")["correct"].agg(["mean", "count"]))

print("\nError rate by position_tier:")
print(test_df.groupby("position_tier")["correct"].agg(["mean", "count"]))

wrong_cases = test_df[~test_df["correct"]].sort_values("model_score", ascending=False)
print("\n3 confident-but-wrong cases:")
print(wrong_cases[["content_id", "client_id", "model_score", "is_declining",
                    "ctr", "avg_position", "days_since_last_update", "content_type"]].head(3).to_string(index=False))

    k  baseline_precision  model_precision  base_rate
0  10                 0.1             0.80   0.516514
1  20                 0.1             0.85   0.516514
2  50                 0.2             0.76   0.516514
Top 10 features by |coefficient| (standardized):
               feature  coefficient
             users_90d    -1.345192
          sessions_90d     1.155801
 days_with_impressions     0.709191
            word_count     0.547541
    days_with_sessions    -0.492363
            char_count    -0.334801
      content_age_days    -0.317042
     scroll_events_90d     0.249670
days_since_last_update     0.158648
           scroll_rate     0.136554

Error rate by content_type:
                        mean  count
content_type                       
comparison article  0.585366    697
keyword article     0.569959   6418

Error rate by position_tier:
                   mean  count
position_tier                 
deep           0.475000    280
page_1         0.594507   3386
page_3_5    

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [12]:
# The model struggles most with top_3 position content, getting it wrong 73.7% of the time versus roughly 53-59% elsewhere, meaning it's least reliable exactly where editors might assume things are safe. It also confidently misclassifies several items from a single client (client_f369cb89fc) as declining when they weren't, suggesting this client's behavior doesn't match the general pattern the model learned from other clients, consistent with the client-to-client variation identified back in the framing stage.
# What it leans on: The top features are users_90d (negative) and sessions_90d (positive), which likely capture engagement quality rather than raw traffic volume, content with many one-time visitors trends toward decline, while content with fewer but more engaged, repeat users does not. days_since_last_update (staleness) appears too, but ranks ninth, meaning engagement signals matter more to the model than the staleness signal the Week-4 baseline relied on entirely.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.